In [9]:
import os

for root, dirs, files in os.walk('../../'):
    for f in files:
        if 'wifi' in f.lower():
            print(os.path.join(root, f))

../../architecture-data-projet\brute\Score de connectivité\indicateur-wifi.csv
../../architecture-data-projet\eda\eda_indicateur_wifi.ipynb


In [15]:
import pandas as pd

FILE = '../../architecture-data-projet/brute/Score de connectivité/indicateur-wifi.csv'

df = pd.read_csv(FILE, sep=None, engine='python', header=4, encoding='latin-1')

# Nettoyage BOM UTF-8
df.columns = df.columns.str.strip().str.replace('\ufeff', '', regex=False)

print(f'Shape : {df.shape}')
print(f'Colonnes : {list(df.columns)}')
df.head()

Shape : (45, 46)
Colonnes : ['Code arrondissement', 'Nom arrondissement', 'Code région', 'Code département', 'Siren EPCI', 'Code commune', 'Logements', 'Établissements', 'Nombre locaux IPE T4 2025 (somme tous OI)', 'Source retenue T4 2025', 'Meilleure estimation des locaux T4 2025', 'Zones très denses', 'OI T4 2025', 'T4 2017', 'T1 2018', 'T2 2018', 'T3 2018', 'T4 2018', 'T1 2019', 'T2 2019', 'T3 2019', 'T4 2019', 'T1 2020', 'T2 2020', 'T3 2020', 'T4 2020', 'T1 2021', 'T2 2021', 'T3 2021', 'T4 2021', 'T1 2022', 'T2 2022', 'T3 2022', 'T4 2022', 'T1 2023', 'T2 2023', 'T3 2023', 'T4 2023', 'T1 2024', 'T2 2024', 'T3 2024', 'T4 2024', 'T1 2025', 'T2 2025', 'T3 2025', 'T4 2025']


,Code arrondissement,Nom arrondissement,Code région,Code département,Siren EPCI,Code commune,Logements,Établissements,Nombre locaux IPE T4 2025 (somme tous OI),Source retenue T4 2025,...,T3 2023,T4 2023,T1 2024,T2 2024,T3 2024,T4 2024,T1 2025,T2 2025,T3 2025,T4 2025
0,13201,Marseille 1er Arrondissement,93,13,200054807,13055,24 176,3 689,31 577,IPE,...,13 216,13 542,13 851,14 182,14 653,15 135,15 425,15 608,15 817,16 122
1,13202,Marseille 2e Arrondissement,93,13,200054807,13055,16 358,2 123,19 788,IPE,...,13 122,13 565,13 676,13 821,14 139,14 294,14 355,14 441,14 839,14 961
2,13203,Marseille 3e Arrondissement,93,13,200054807,13055,27 774,1 215,30 742,IPE,...,22 140,22 752,23 345,24 594,25 184,26 429,26 830,27 167,27 304,27 531
3,13204,Marseille 4e Arrondissement,93,13,200054807,13055,29 927,1 281,33 951,IPE,...,29 442,29 569,29 789,29 971,30 278,30 730,30 842,31 009,31 303,31 358
4,13205,Marseille 5e Arrondissement,93,13,200054807,13055,31 123,1 113,34 806,IPE,...,30 432,30 559,30 681,30 896,31 273,31 847,32 061,32 264,32 455,32 736


In [16]:
print(df.dtypes)
print(f'\nValeurs manquantes par colonne :')
print(df.isnull().sum()[df.isnull().sum() > 0])
print(f'\nTotal lignes : {len(df)}')
print(f'\nCodes arrondissement uniques : {sorted(df["Code arrondissement"].unique())}')

Code arrondissement                           int64
Nom arrondissement                           object
Code région                                   int64
Code département                              int64
Siren EPCI                                    int64
Code commune                                  int64
Logements                                    object
Établissements                               object
Nombre locaux IPE T4 2025 (somme tous OI)    object
Source retenue T4 2025                       object
Meilleure estimation des locaux T4 2025      object
Zones très denses                             int64
OI T4 2025                                   object
T4 2017                                      object
T1 2018                                      object
T2 2018                                      object
T3 2018                                      object
T4 2018                                      object
T1 2019                                      object
T2 2019     

In [17]:
# Filtrer Paris uniquement
df_paris = df[df['Code arrondissement'].between(75101, 75120)].copy()
print(f'Lignes Paris : {len(df_paris)}')

# Colonnes trimestrielles
cols_trim = [c for c in df.columns if any(t in c for t in ['T1','T2','T3','T4']) and '2025' in c or 
             any(t in c for t in ['T1','T2','T3','T4']) and c != 'Source retenue T4 2025']
cols_trim = [c for c in df.columns if c.startswith('T') or (len(c) > 2 and c[2] == ' ')]

# Plus simple : toutes les colonnes sauf les métadonnées
cols_numeriques = ['Logements', 'Établissements', 'Nombre locaux IPE T4 2025 (somme tous OI)',
                   'Meilleure estimation des locaux T4 2025'] + \
                  [c for c in df.columns if c.startswith('T')]

for col in cols_numeriques:
    if col in df_paris.columns:
        df_paris[col] = df_paris[col].astype(str).str.replace(' ', '').str.replace('\xa0', '')
        df_paris[col] = pd.to_numeric(df_paris[col], errors='coerce')

print(f'\nValeurs manquantes après nettoyage :')
print(df_paris.isnull().sum()[df_paris.isnull().sum() > 0])
print(f'\nTypes après nettoyage :')
print(df_paris[cols_numeriques].dtypes)

Lignes Paris : 20

Valeurs manquantes après nettoyage :
Series([], dtype: int64)

Types après nettoyage :
Logements                                    int64
Établissements                               int64
Nombre locaux IPE T4 2025 (somme tous OI)    int64
Meilleure estimation des locaux T4 2025      int64
T4 2017                                      int64
T1 2018                                      int64
T2 2018                                      int64
T3 2018                                      int64
T4 2018                                      int64
T1 2019                                      int64
T2 2019                                      int64
T3 2019                                      int64
T4 2019                                      int64
T1 2020                                      int64
T2 2020                                      int64
T3 2020                                      int64
T4 2020                                      int64
T1 2021                    

In [18]:
cols_cles = ['Logements', 'Établissements', 
             'Meilleure estimation des locaux T4 2025',
             'T4 2017', 'T4 2020', 'T4 2022', 'T4 2025']

print(df_paris[['Code arrondissement', 'Nom arrondissement'] + cols_cles].to_string(index=False))
print('\n--- Stats descriptives ---')
print(df_paris[cols_cles].describe().round(0))

 Code arrondissement       Nom arrondissement  Logements  Établissements  Meilleure estimation des locaux T4 2025  T4 2017  T4 2020  T4 2022  T4 2025
               75101 Paris 1er Arrondissement      13872            7444                                    24615    17455    20035    21475    22342
               75102  Paris 2e Arrondissement      17332            7326                                    31514    21504    25823    28120    29371
               75103  Paris 3e Arrondissement      26407            4907                                    37631    27703    31106    32909    35008
               75104  Paris 4e Arrondissement      23031            3803                                    31728    24511    27623    29248    30373
               75105  Paris 5e Arrondissement      39882            4718                                    51953    41364    45221    46948    48826
               75106  Paris 6e Arrondissement      31644            6265                            

In [19]:
# Calcul du taux de raccordement fibre T4 2025
df_paris['taux_fibre_T4_2025'] = (df_paris['T4 2025'] / df_paris['Meilleure estimation des locaux T4 2025'] * 100).round(2)

# Évolution T4 2017 → T4 2025
df_paris['taux_fibre_T4_2017'] = (df_paris['T4 2017'] / df_paris['Meilleure estimation des locaux T4 2025'] * 100).round(2)

cols_affich = ['Code arrondissement', 'Nom arrondissement', 
               'Meilleure estimation des locaux T4 2025',
               'T4 2017', 'T4 2025',
               'taux_fibre_T4_2017', 'taux_fibre_T4_2025']

print(df_paris[cols_affich].sort_values('taux_fibre_T4_2025', ascending=False).to_string(index=False))

 Code arrondissement       Nom arrondissement  Meilleure estimation des locaux T4 2025  T4 2017  T4 2025  taux_fibre_T4_2017  taux_fibre_T4_2025
               75116 Paris 16e Arrondissement                                   144971   129234   141979               89.14               97.94
               75108  Paris 8e Arrondissement                                    54940    44112    53292               80.29               97.00
               75117 Paris 17e Arrondissement                                   138677   116291   134255               83.86               96.81
               75107  Paris 7e Arrondissement                                    57536    46738    55604               81.23               96.64
               75106  Paris 6e Arrondissement                                    46155    35341    44508               76.57               96.43
               75113 Paris 13e Arrondissement                                   124178   104034   119750               83.78      

In [20]:
# Colonnes trimestrielles uniquement
cols_t = [c for c in df_paris.columns if c.startswith('T') and c != 'Zones très denses']

# Focus sur les 3 arrondissements extrêmes
arr_focus = [75116, 75119, 75120, 75101]
df_focus = df_paris[df_paris['Code arrondissement'].isin(arr_focus)][['Code arrondissement'] + cols_t]

print("Évolution trimestrielle (locaux raccordés) :")
print(df_focus.set_index('Code arrondissement').T.to_string())

# Vérif anomalie : est-ce que T4 2025 < T4 2024 pour certains ?
df_paris['delta_2024_2025'] = df_paris['T4 2025'] - df_paris['T4 2024']
print('\nDelta T4 2024 → T4 2025 (négatif = anomalie) :')
print(df_paris[['Code arrondissement', 'Nom arrondissement', 'T4 2024', 'T4 2025', 'delta_2024_2025']].sort_values('delta_2024_2025').to_string(index=False))

Évolution trimestrielle (locaux raccordés) :
Code arrondissement  75101   75116   75119   75120
T4 2017              17455  129234   96669  104216
T1 2018              17757  131021   97410  105145
T2 2018              17878  132717   97998  105452
T3 2018              18237  133482   98300  107708
T4 2018              18253  134426   99083  107436
T1 2019              18341  135261   99585  107861
T2 2019              18485  135685  100517  109053
T3 2019              18695  136006  101452  109480
T4 2019              19124  136492  102793  110570
T1 2020              19315  136862  103088  110949
T2 2020              19491  136154  103884  111175
T3 2020              19629  136642  104504  112002
T4 2020              20035  137196  105480  114955
T1 2021              20592  137618  106019  115705
T2 2021              20768  138080  106773  116101
T3 2021              20853  138242  107249  116350
T4 2021              21007  138849  107537  116649
T1 2022              21085  139407  1

In [21]:
# Résumé EDA final
print("=== RÉSUMÉ EDA indicateur-wifi ===\n")
print(f"Lignes Paris retenues     : {len(df_paris)} (75101 → 75120)")
print(f"Valeurs manquantes        : 0")
print(f"Colonnes trimestrielles   : {len(cols_t)} (T4 2017 → T4 2025)")
print(f"Taux fibre min            : {df_paris['taux_fibre_T4_2025'].min()}% ({df_paris.loc[df_paris['taux_fibre_T4_2025'].idxmin(), 'Nom arrondissement']})")
print(f"Taux fibre max            : {df_paris['taux_fibre_T4_2025'].max()}% ({df_paris.loc[df_paris['taux_fibre_T4_2025'].idxmax(), 'Nom arrondissement']})")
print(f"Taux fibre moyen Paris    : {df_paris['taux_fibre_T4_2025'].mean().round(2)}%")
print(f"Tous arrondissements > 85%: {(df_paris['taux_fibre_T4_2025'] > 85).all()}")

# Table silver prête
df_silver = df_paris[['Code arrondissement', 'Nom arrondissement',
                       'Logements', 'Établissements',
                       'Meilleure estimation des locaux T4 2025',
                       'T4 2025', 'taux_fibre_T4_2025']].copy()

df_silver.columns = ['code_arrondissement', 'nom_arrondissement',
                     'nb_logements', 'nb_etablissements',
                     'locaux_total', 'locaux_fibres_T4_2025',
                     'taux_fibre_pct']

print(f"\nAperçu table silver :")
print(df_silver.to_string(index=False))

=== RÉSUMÉ EDA indicateur-wifi ===

Lignes Paris retenues     : 20 (75101 → 75120)
Valeurs manquantes        : 0
Colonnes trimestrielles   : 33 (T4 2017 → T4 2025)
Taux fibre min            : 88.14% (Paris 19e Arrondissement)
Taux fibre max            : 97.94% (Paris 16e Arrondissement)
Taux fibre moyen Paris    : 94.35%
Tous arrondissements > 85%: True

Aperçu table silver :
 code_arrondissement       nom_arrondissement  nb_logements  nb_etablissements  locaux_total  locaux_fibres_T4_2025  taux_fibre_pct
               75101 Paris 1er Arrondissement         13872               7444         24615                  22342           90.77
               75102  Paris 2e Arrondissement         17332               7326         31514                  29371           93.20
               75103  Paris 3e Arrondissement         26407               4907         37631                  35008           93.03
               75104  Paris 4e Arrondissement         23031               3803         31728 